In [6]:
import pandas as pd

match_summary = pd.read_csv(
    r"C:\Users\ckcha\OneDrive\Desktop\CSK_Dashboard\Clean_Data\match_summary.csv"
)


match_summary.columns.tolist()

#Create Winning DNA Dataset

match_summary["win_flag"].value_counts()


#Create Features
features = [
    "pp_runs",
    "pp_wickets",
    "middle_runs",
    "middle_wickets",
    "death_runs",
    "death_wickets",
    "boundaries",
    "dot_balls"
]

target = "win_flag"

#Train Test Split

from sklearn.model_selection import train_test_split

X = match_summary[features]
y = match_summary[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#Random Forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42
)

rf.fit(X_train, y_train)

#Accuracy

from sklearn.metrics import accuracy_score

y_pred = rf.predict(X_test)

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:", accuracy)

importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

importance.to_csv(
    r"C:\Users\ckcha\OneDrive\Desktop\CSK_Dashboard\ML_Models\winning_dna.csv",
    index=False
)


Accuracy: 0.7413441955193483
          Feature  Importance
5   death_wickets    0.164330
7       dot_balls    0.150949
4      death_runs    0.133611
0         pp_runs    0.121637
6      boundaries    0.121487
2     middle_runs    0.112172
3  middle_wickets    0.110220
1      pp_wickets    0.085594


In [20]:
#Failure DNA

winning = match_summary[
    match_summary["win_flag"] == 1
]

losing = match_summary[
    match_summary["win_flag"] == 0
]


#Create Failure DNA Table
failure_dna = pd.DataFrame({

    "Winning": winning[
        [
            "pp_runs",
            "pp_wickets",
            "middle_runs",
            "middle_wickets",
            "death_runs",
            "death_wickets",
            "boundaries",
            "dot_balls"
        ]
    ].mean(),

    "Losing": losing[
        [
            "pp_runs",
            "pp_wickets",
            "middle_runs",
            "middle_wickets",
            "death_runs",
            "death_wickets",
            "boundaries",
            "dot_balls"
        ]
    ].mean()
})

#Gap Analysis

failure_dna["Gap"] = (
    failure_dna["Winning"]
    -
    failure_dna["Losing"]
)

print(failure_dna)

failure_dna.to_csv(
    r"C:\Users\ckcha\OneDrive\Desktop\CSK_Dashboard\ML_Models\failure_dna.csv"
)

                  Winning     Losing       Gap
pp_runs         50.553557  44.658259  5.895298
pp_wickets       1.112837   1.751831 -0.638993
middle_runs     72.632052  65.362897  7.269156
middle_wickets   1.846280   2.688365 -0.842085
death_runs      43.177433  41.700570  1.476863
death_wickets    1.606705   2.684296 -1.077591
boundaries      21.505315  18.153784  3.351531
dot_balls       37.170074  43.457282 -6.287209


In [26]:
# SCORE PREDICTION MODEL

import pandas as pd

clean_ball = pd.read_csv(
    r"C:\Users\ckcha\OneDrive\Desktop\CSK_Dashboard\Data\clean_ball_data.csv"
)

# -------------------------------------
# Final Score
# -------------------------------------

score_df = clean_ball.groupby(
    ["match_id","innings"]
).agg(
    final_score=("total_runs","sum")
).reset_index()

# -------------------------------------
# Powerplay Features
# -------------------------------------

pp = clean_ball[
    clean_ball["over_number"] <= 6
]

pp_features = pp.groupby(
    ["match_id","innings"]
).agg(
    pp_runs=("total_runs","sum"),
    pp_wickets=("wicket_flag","sum")
).reset_index()

# -------------------------------------
# Score After 10 Overs
# -------------------------------------

first10 = clean_ball[
    clean_ball["over_number"] <= 10
]

score10 = first10.groupby(
    ["match_id","innings"]
).agg(
    score_after_10=("total_runs","sum")
).reset_index()

# -------------------------------------
# Middle Overs Features
# -------------------------------------

middle = clean_ball[
    (clean_ball["over_number"] >= 7)
    &
    (clean_ball["over_number"] <= 15)
]

middle_features = middle.groupby(
    ["match_id","innings"]
).agg(
    middle_runs=("total_runs","sum"),
    middle_wickets=("wicket_flag","sum")
).reset_index()

# -------------------------------------
# Boundary & Dot Ball Features
# -------------------------------------

boundary_features = clean_ball.groupby(
    ["match_id","innings"]
).agg(
    boundaries=("boundary_flag","sum"),
    dot_balls=("dot_ball_flag","sum")
).reset_index()

# -------------------------------------
# Merge Everything
# -------------------------------------

score_prediction = pp_features.merge(
    middle_features,
    on=["match_id","innings"]
)

score_prediction = score_prediction.merge(
    score10,
    on=["match_id","innings"]
)

score_prediction = score_prediction.merge(
    boundary_features,
    on=["match_id","innings"]
)

score_prediction = score_prediction.merge(
    score_df,
    on=["match_id","innings"]
)

print(score_prediction.shape)

# -------------------------------------
# Train Model
# -------------------------------------

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

features = [
    "pp_runs",
    "pp_wickets",
    "middle_runs",
    "middle_wickets",
    "score_after_10",
    "boundaries",
    "dot_balls"
]

target = "final_score"

X = score_prediction[features]
y = score_prediction[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# -------------------------------------
# Improved Random Forest
# -------------------------------------

model = RandomForestRegressor(
    n_estimators=500,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

model.fit(X_train, y_train)

# -------------------------------------
# Prediction
# -------------------------------------

preds = model.predict(X_test)

mae = mean_absolute_error(
    y_test,
    preds
)

print("MAE:", mae)

# -------------------------------------
# Save Dataset
# -------------------------------------

score_prediction.to_csv(
    r"C:\Users\ckcha\OneDrive\Desktop\CSK_Dashboard\ML_Models\score_prediction_dataset.csv",
    index=False
)

(2406, 10)
MAE: 8.39522903861576
